# 5. Content-Based Kindle Recommendation

This section develops and evaluates a content-based recommender for Kindle books. Each book is represented using textual metadata, including its title, categories, features, description, and optionally author information.

A user profile is constructed by aggregating the representations of books in the
user’s training history. Candidate books are then ranked according to their cosine similarity to the user profile.

The experiments follow a progressive complexity ladder:

1. Popularity baseline
2. TF-IDF with cosine similarity
3. Field-weighted TF-IDF with cosine similarity
4. TF-IDF with SVD and cosine similarity

All model selection and hyperparameter tuning are conducted on the validation set. The test set is evaluated only once after the final configuration is selected.

## 5.1 Setup and Configuration

### 5.11 Environment setup



We support colab environment and local environment, this code can judge the running environment.

### 5.12 Experimental Configuration for Conten-Based
The following configuration sets the random seeds, file paths, recommendation cutoff, candidate-set size, text-processing settings, and model hyperparameters. This was used throughout the content-based experiments.

| Configuration | Value | Purpose |
|---|---:|---|
| Description truncation | 1,500 characters | Controls text length and memory use |
| Basic TF-IDF dimensions | 32,768 | Defines the hashing feature space |
| Weighted TF-IDF n-grams | (1, 2) | Uses unigram and bigram features |
| Sublinear TF scaling | Enabled | Reduces the influence of repeated terms |
| SVD vocabulary size | 20,000 | Limits the explicit TF-IDF vocabulary |
| Minimum document frequency | 3 | Removes very rare terms |
| Default SVD dimensions | 128 | Dense representation size |
| Recommendation cutoff | 10 | Returns and evaluates Top-10 lists |
| Candidate pool | 50 | Number of content candidates retrieved |
| Nearest neighbours | 50 | Number of neighbours queried from the index |
| Verified-purchase multiplier | 1.0 | Controls interaction weighting |
| Rating scale maximum | 5.0 | Normalises rating weights |
| Primary metric | NDCG@10 | Selects models and hyperparameters |
| Model/evaluation seeds | 2026 / 2026 | Ensures reproducibility |
| Diagnostic-user sample | 5,000 | Used for catalogue-level diagnostics |

In [74]:
from __future__ import annotations

import hashlib
import json
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

CONFIG = {
    "data_dir_candidates": PROJECT_ROOT / "processed_kindle",
    "output_dir": "outputs/content_based",
    # basic TFIDF
    "description_truncate_chars": 1500,
    "basic_n_features": 2 ** 15,

    # Weighted-TFIDF
    "basic_field_weights": {
        "title": 1.0,
        "categories": 1.0,
        "features": 1.0,
        "description": 1.0,
        "author": 0.0,
    },
    "weighted_field_weight_candidates": {
        "title2_categories2": {"title": 2.0, "categories": 2.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title3_categories2": {"title": 3.0, "categories": 2.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories3": {"title": 2.0, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories2_features1_5": {"title": 2.0, "categories": 2.0, "features": 1.5, "description": 1.0, "author": 0.0},
        "title2_categories2_5": {"title": 2.0, "categories": 2.5, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories3_5": {"title": 2.0, "categories": 3.5, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories4": {"title": 2.0, "categories": 4.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title1_5_categories3": {"title": 1.5, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_5_categories3": {"title": 2.5, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_25_categories2_75": {"title": 2.25, "categories": 2.75, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_25_categories3": {"title": 2.25, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_5_categories2_75": {"title": 2.5, "categories": 2.75, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_5_categories3_25": {"title": 2.5, "categories": 3.25, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_75_categories3": {"title": 2.75, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
    },

    "weighted_tfidf_ngram_range": (1, 2),
    "weighted_tfidf_sublinear_tf": True,

    # Variant 1 (TF-IDF + SVD)
    "svd_max_features": 20_000,
    "svd_min_df": 3,
    "svd_n_components": 128,

    # Variant 1b (Field-Weighted TF-IDF + SVD)
    "weighted_svd_dim_candidates": [128, 256, 512],

    # Shared recommendation parameters
    "n_neighbors_index": 50,
    "top_k": 10,
    "candidate_pool": 50,

    # User-profile weighting.
    "verified_purchase_multiplier": 1.0,
    "rating_scale_max": 5.0,

    # Shared evaluation protocol.
    "eval_k": 10,
    "primary_metric": "NDCG@10",
    "model_seed": 2026,
    "evaluation_seed": 2026,
    "diagnostic_n_users": 5000,
}

DATA_DIR = DATA_ROOT
OUTPUT_DIR = PROJECT_ROOT / CONFIG["output_dir"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def progress(message: str, done: bool = False) -> None:
    end = "\n" if done else "\r"
    print(message.ljust(100), end=end, flush=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


seed_everything(CONFIG["model_seed"])
print(
    f"data_dir={DATA_DIR.resolve()}\n"
    f"model_seed={CONFIG['model_seed']}, evaluation_seed={CONFIG['evaluation_seed']}, "
    f"primary_metric={CONFIG['primary_metric']}"
)


data_dir=/Users/lumi/PycharmProjects/PythonProject/comp9727/teamProject/processed_kindle
model_seed=2026, evaluation_seed=2026, primary_metric=NDCG@10


# 5.2 Data Loading

This section validates and loads the processed data required by the
content-based recommender. It also prepares the shared item-text representation
used by the TF-IDF, SVD.

| Function | Purpose |
|---|---|
| `require_files()` | Checks that all required processed files exist before loading begins |
| `normalise_split_name()` | Standardises split names such as `val` and `validation` |
| `load_mappings()` | Loads user and item ID-to-index mapping files |
| `prepare_item_frame()` | Cleans text fields, fills missing values, normalises category separators, and truncates descriptions |
| `build_unweighted_content_text()` | Concatenates title, categories, features, and description once each into `content_text` |
| `load_items()` | Loads and merges item content, author, and rating metadata by `item_idx` |
| `load_interactions()` | Loads all interactions or filters them by data split |
| `load_eval_positives()` | Loads held-out positive items for validation or testing |
| `load_eval_negatives()` | Loads each user's fixed negative candidates for sampled evaluation |

In [116]:
REQUIRED_FILES = [
    "user2idx.json", "item2idx.json", "idx2user.json", "idx2item.json",
    "items_metadata_clean.csv", "item_metadata.csv", "interactions_clean.csv",
    "val.csv", "test.csv", "val_negatives.csv", "test_negatives.csv",
]

def require_files(data_dir: Path) -> None:
    missing = [name for name in REQUIRED_FILES if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing {len(missing)} required file(s) under {data_dir.resolve()}: {missing}"
        )


def normalise_split_name(split: str) -> str:
    aliases = {"val": "validation", "validation": "validation", "test": "test", "train": "train"}
    if split not in aliases:
        raise ValueError(f"Unknown split={split!r}; expected train, validation/val, or test")
    return aliases[split]


def load_mappings(data_dir: Path) -> dict:
    mappings = {}
    for name in ["user2idx", "item2idx", "idx2user", "idx2item"]:
        with open(data_dir / f"{name}.json", "r", encoding="utf-8") as f:
            mappings[name] = json.load(f)
    return mappings


ITEM_TEXT_FIELDS = ("title", "categories", "features", "description", "author")

# Clean text
def prepare_item_frame(items_df: pd.DataFrame, description_truncate_chars: int) -> pd.DataFrame:
    # Adding truncate
    frame = items_df.copy()
    for column in ITEM_TEXT_FIELDS:
        if column not in frame.columns:
            frame[column] = ""
        frame[column] = frame[column].fillna("").astype(str)
    frame["categories"] = frame["categories"].str.replace(";", " ", regex=False)
    frame["description"] = frame["description"].str.slice(0, description_truncate_chars)
    return frame


def build_unweighted_content_text(
    items_df: pd.DataFrame,
    description_truncate_chars: int,
) -> pd.Series:

    frame = prepare_item_frame(items_df, description_truncate_chars)
    return (
        frame["title"] + " "
        + frame["categories"] + " "
        + frame["features"] + " "
        + frame["description"]
    ).str.replace(r"\s+", " ", regex=True).str.strip()


def load_items(data_dir: Path, description_truncate_chars: int) -> pd.DataFrame:
    content_df = pd.read_csv(data_dir / "items_metadata_clean.csv")
    stats_df = pd.read_csv(data_dir / "item_metadata.csv")[
        ["item_idx", "author", "average_rating", "rating_number"]
    ]
    items_df = content_df.merge(stats_df, on="item_idx", how="left", validate="one_to_one")

    for col in ["title", "categories", "features", "description", "author"]:
        items_df[col] = items_df[col].fillna("")

    items_df["content_text"] = build_unweighted_content_text(items_df, description_truncate_chars)
    return items_df.sort_values("item_idx").reset_index(drop=True)


def load_interactions(data_dir: Path, split: str | None = None) -> pd.DataFrame:
    df = pd.read_csv(data_dir / "interactions_clean.csv")
    if split is not None:
        split = normalise_split_name(split)
        df = df[df["split"] == split].reset_index(drop=True)
    return df


def load_eval_positives(data_dir: Path, split: str) -> pd.DataFrame:
    split = normalise_split_name(split)
    fname = "val.csv" if split == "validation" else "test.csv"
    return pd.read_csv(data_dir / fname)


def load_eval_negatives(data_dir: Path, split: str) -> dict[int, list[int]]:
    split = normalise_split_name(split)
    fname = "val_negatives.csv" if split == "validation" else "test_negatives.csv"
    df = pd.read_csv(data_dir / fname)
    return {
        int(row.user_idx): [int(value) for value in row.neg_items.split()]
        for row in df.itertuples()
    }


The following function checks index alignment, candidate-set validity, and
train, validation, test leakage before any model is fitted.

In [76]:
def audit_protocol(
    mappings: dict,
    items_df: pd.DataFrame,
    train_interactions: pd.DataFrame,
    validation_interactions: pd.DataFrame,
    val_pos: pd.DataFrame,
    test_pos: pd.DataFrame,
    val_neg: dict,
    test_neg: dict,
) -> None:
    failures = []
    n_items = len(mappings["idx2item"])

    if items_df["item_idx"].tolist() != list(range(n_items)):
        failures.append(
            f"items_df item_idx is not the continuous range 0..{n_items - 1}; "
            f"actual rows={len(items_df)}"
        )

    train_sets = train_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
    validation_map = dict(
        zip(
            validation_interactions["user_idx"].astype(int),
            validation_interactions["item_idx"].astype(int),
        )
    )
    val_target_map = dict(zip(val_pos["user_idx"].astype(int), val_pos["pos_idx"].astype(int)))
    test_target_map = dict(zip(test_pos["user_idx"].astype(int), test_pos["pos_idx"].astype(int)))

    if validation_map != val_target_map:
        failures.append("validation rows in interactions_clean.csv do not match val.csv")

    for name, pos_df, neg_dict in [
        ("validation", val_pos, val_neg),
        ("test", test_pos, test_neg),
    ]:
        pos_users = set(pos_df["user_idx"].astype(int))
        neg_users = set(neg_dict)
        if pos_users != neg_users:
            failures.append(
                f"{name}: positive and negative user sets differ "
                f"(missing={len(pos_users - neg_users)}, extra={len(neg_users - pos_users)})"
            )

        for row in pos_df.itertuples():
            user, positive = int(row.user_idx), int(row.pos_idx)
            negatives = neg_dict.get(user, [])
            if len(negatives) != 100 or len(set(negatives)) != 100:
                failures.append(
                    f"{name} user={user}: expected 100 distinct negatives; "
                    f"got {len(negatives)} ({len(set(negatives))} distinct)"
                )
            if positive in negatives:
                failures.append(f"{name} user={user}: positive item appears in negatives")
            if any(item < 0 or item >= n_items for item in [positive, *negatives]):
                failures.append(f"{name} user={user}: candidate item is outside 0..{n_items - 1}")

            # Fixed negatives must not include any known positive for this user.
            all_known_positives = set(train_sets.get(user, set()))
            all_known_positives.update({val_target_map[user], test_target_map[user]})
            if set(negatives) & all_known_positives:
                failures.append(f"{name} user={user}: negatives overlap the user's known positives")

    for user, val_item in val_target_map.items():
        if val_item in train_sets.get(user, set()):
            failures.append(f"validation user={user}: held-out item appears in training history")
    for user, test_item in test_target_map.items():
        if test_item in train_sets.get(user, set()) or test_item == val_target_map[user]:
            failures.append(f"test user={user}: held-out item leaks/repeats in earlier history")

    if failures:
        raise AssertionError(
            f"Protocol audit failed with {len(failures)} issue(s); first five:\n"
            + "\n".join(failures[:5])
        )

    print(
        f"Protocol audit passed: {n_items:,} items, "
        f"validation={len(val_pos):,} users, test={len(test_pos):,} users, "
        f"train_interactions={len(train_interactions):,}. "
        "Validation context=train; test context=train+validation."
    )


Load and Prepare the Shared Evaluation Data

In [77]:
require_files(DATA_DIR)

mappings = load_mappings(DATA_DIR)
items_df = load_items(DATA_DIR, CONFIG["description_truncate_chars"])
all_interactions = load_interactions(DATA_DIR)
train_interactions = all_interactions[all_interactions["split"] == "train"].reset_index(drop=True)
validation_interactions = all_interactions[
    all_interactions["split"] == "validation"
].reset_index(drop=True)

# Chronological evaluation contexts:
#   validation profile = train
#   test profile       = train + validation interaction
test_profile_interactions = pd.concat(
    [train_interactions, validation_interactions],
    ignore_index=True,
)

n_users = len(mappings["idx2user"])
n_items = len(mappings["idx2item"])

val_positives = load_eval_positives(DATA_DIR, "validation")
test_positives = load_eval_positives(DATA_DIR, "test")
val_negatives = load_eval_negatives(DATA_DIR, "validation")
test_negatives = load_eval_negatives(DATA_DIR, "test")

audit_protocol(
    mappings,
    items_df,
    train_interactions,
    validation_interactions,
    val_positives,
    test_positives,
    val_negatives,
    test_negatives,
)

# Shared popularity is learned only from training interactions, exactly as in the
# sequential notebook. Metadata rating_number is retained for display only.
train_item_counts = (
    train_interactions.groupby("item_idx")
    .size()
    .reindex(range(n_items), fill_value=0)
    .astype(np.int64)
    .to_numpy()
)
items_df["train_interaction_count"] = train_item_counts
items_df["popularity_score"] = np.log1p(train_item_counts)

train_interacted_items = (
    train_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
)
test_interacted_items = (
    test_profile_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
)

description_missing_rate = (items_df["description"].str.len() == 0).mean()
print(
    f"items={n_items:,}, users={n_users:,}, train={len(train_interactions):,}, "
    f"validation_context_additions={len(validation_interactions):,}, "
    f"description_missing={description_missing_rate:.1%}"
)


Protocol audit passed: 81,322 items, validation=64,867 users, test=64,867 users, train_interactions=884,136. Validation context=train; test context=train+validation.
items=81,322, users=64,867, train=884,136, validation_context_additions=64,867, description_missing=39.8%


## 5.3 Popularity Baseline

The popularity model is a non-personalised baseline used to check whether the content-based recommender is actually more effective.

Books are ranked by their interaction counts in the training set:
popularity(i)=log(1+ci), where ci is the number of training interactions for book i. (for above code : np.log1p(ci))

In [78]:
class PopularityRecommender:

    def __init__(self, items_df: pd.DataFrame, interacted_items: dict[int, set[int]]):
        self.items_df = items_df.set_index("item_idx")
        self.interacted_items = interacted_items
        self.ranked_items = (
            items_df.sort_values(
                ["popularity_score", "item_idx"],
                ascending=[False, True],
            )
            ["item_idx"]
            .astype(int)
            .tolist()
        )

    def recommend(self, user_idx: int, k: int | None = None, candidate_pool: int | None = None):
        k = k or CONFIG["top_k"]
        seen = self.interacted_items.get(user_idx, set())
        item_ids = [item for item in self.ranked_items if item not in seen][:k]
        return [
            {
                "item_idx": item,
                "reason": "popular",
                "similarity": 0.0,
                "title": self.items_df.loc[item, "title"],
            }
            for item in item_ids
        ]

    def score_candidates(self, user_idx: int, candidate_item_idxs: list[int]) -> np.ndarray:
        return self.items_df.loc[candidate_item_idxs, "popularity_score"].to_numpy(dtype=float)


### 5.4 Basic TF-IDF Encoder with Cosine Similarity

This encoder converts each book's `content_text` into a sparse TF-IDF vector. `HashingVectorizer` first maps the text into a fixed-size term-count space, and `TfidfTransformer` then assigns TF-IDF weights to the hashed features. After the item vectors are created, a brute-force nearest-neighbour index is built using cosine distance. Given a user-profile vector, `query_topk()` returns the books with the most similar content representations.

Newly added books can be transformed directly from their text metadata using the
existing TF-IDF space, without refitting the full encoder.

| Method | Purpose |
|---|---|
| `fit_transform()` | Builds TF-IDF vectors for all existing books |
| `build_index()` | Creates the cosine nearest-neighbour index |
| `query_topk()` | Retrieves the Top-K most similar books |
| `transform_new_items()` | Encodes newly added books without refitting |

In [79]:
class TFIDFHashingVectorizer:
    def __init__(self, n_features: int):
        self.hv = HashingVectorizer(n_features=n_features, stop_words="english",
                                     alternate_sign=False, norm=None)
        self.tt = TfidfTransformer()
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None

    def fit_transform(self, items_df: pd.DataFrame):
        items_df = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = items_df["item_idx"].to_numpy()
        counts = self.hv.transform(items_df["content_text"])
        self.item_matrix = self.tt.fit_transform(counts)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors, k: int):
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, content_texts: list[str]):
        counts = self.hv.transform(content_texts)
        return self.tt.transform(counts)


### 5.5 Field-Weighted TF-IDF with Cosine Similarity

This variant assigns separate weights to title, categories, features,
description, and author. Each field is hashed independently, multiplied by its
field weight, and then combined before TF-IDF transformation.

| Method | Purpose |
|---|---|
| `_weighted_counts()` | Vectorises each metadata field, applies its field weight, and combines the weighted count matrices |
| `fit_transform()` | Builds the weighted TF-IDF representation for all books |
| `build_index()` | Creates a cosine-distance nearest-neighbour index |
| `query_topk()` | Retrieves the Top-K items most similar to a query vector |
| `transform_new_items()` | Transforms newly added books using the existing TF-IDF space without refitting |


In [80]:
class FieldWeightedTFIDFVectorizer:

    def __init__(
        self,
        n_features: int,
        field_weights: dict[str, float],
        description_truncate_chars: int,
        ngram_range: tuple[int, int] = (1, 1),
        sublinear_tf: bool = False,
    ):
        unknown = set(field_weights) - set(ITEM_TEXT_FIELDS)
        if unknown:
            raise ValueError(f"Unknown item text fields: {sorted(unknown)}")
        if not any(float(weight) > 0 for weight in field_weights.values()):
            raise ValueError("At least one field weight must be positive")
        if any(float(weight) < 0 for weight in field_weights.values()):
            raise ValueError("Field weights cannot be negative")

        self.field_weights = {
            field: float(field_weights.get(field, 0.0))
            for field in ITEM_TEXT_FIELDS
        }
        self.description_truncate_chars = int(description_truncate_chars)
        self.ngram_range = tuple(ngram_range)
        self.sublinear_tf = bool(sublinear_tf)
        self.hv = HashingVectorizer(
            n_features=n_features,
            stop_words="english",
            alternate_sign=False,
            norm=None,
            ngram_range=self.ngram_range,
        )
        self.tt = TfidfTransformer(norm="l2", sublinear_tf=self.sublinear_tf)
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None

    def _weighted_counts(self, items_df: pd.DataFrame):
        frame = prepare_item_frame(items_df, self.description_truncate_chars)
        weighted_counts = None
        for field, weight in self.field_weights.items():
            if weight <= 0:
                continue
            field_counts = self.hv.transform(frame[field])
            if weight != 1.0:
                field_counts = field_counts.multiply(weight)
            weighted_counts = (
                field_counts if weighted_counts is None else weighted_counts + field_counts
            )
        if weighted_counts is None:
            raise RuntimeError("No positive field weight was available")
        return weighted_counts.tocsr()

    def fit_transform(self, items_df: pd.DataFrame):
        ordered = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = ordered["item_idx"].to_numpy()
        counts = self._weighted_counts(ordered)
        self.item_matrix = self.tt.fit_transform(counts).astype(np.float32)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        n_neighbors = min(int(n_neighbors), int(self.item_matrix.shape[0]))
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors, k: int):
        k = min(int(k), int(self.item_matrix.shape[0]))
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, new_items: pd.DataFrame):
        counts = self._weighted_counts(new_items)
        return self.tt.transform(counts).astype(np.float32)


### 5.6 TF-IDF with Truncated SVD and Cosine Similarity

This variant first represents each book using TF-IDF and then applies
`TruncatedSVD` to reduce the high-dimensional sparse vectors into a compact
latent semantic space. The reduced item vectors are L2-normalised before
similarity calculation.


And others, which same as 5.5 and 5.4.

In [81]:
class SVDContentVectorizer:
    def __init__(self, max_features: int, min_df: int, n_components: int):
        self.tfidf = TfidfVectorizer(max_features=max_features, min_df=min_df, stop_words="english")
        self.svd = TruncatedSVD(n_components=n_components, random_state=CONFIG["model_seed"])
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None
        self.raw_tfidf_matrix = None
        self.feature_names = None

    def fit_transform(self, items_df: pd.DataFrame) -> np.ndarray:
        items_df = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = items_df["item_idx"].to_numpy()
        tfidf_matrix = self.tfidf.fit_transform(items_df["content_text"])
        self.raw_tfidf_matrix = tfidf_matrix
        self.feature_names = self.tfidf.get_feature_names_out()
        reduced = self.svd.fit_transform(tfidf_matrix).astype(np.float32)
        self.item_matrix = normalize(reduced)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors: np.ndarray, k: int):
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, content_texts: list[str]):
        tfidf_vec = self.tfidf.transform(content_texts)
        reduced = self.svd.transform(tfidf_vec).astype(np.float32)
        return normalize(reduced)


### 5.7 Field-Weighted TF-IDF with Truncated SVD and Cosine Similarity
This model combines field-weighted TF-IDF with dimensionality reduction.

Using `FieldWeightedTFIDFVectorizer` to process metadata fields, apply their numerical weights, and produce a weighted TF-IDF representation.

Then applied `TruncatedSVD` to the weighted sparse matrix to project the item vectors into a lower-dimensional latent semantic space. The reduced vectors are L2-normalised before similarity calculation.

In [82]:
class FieldWeightedSVDContentVectorizer:
    def __init__(
        self,
        n_features: int,
        field_weights: dict[str, float],
        description_truncate_chars: int,
        n_components: int,
        ngram_range: tuple[int, int] = (1, 1),
        sublinear_tf: bool = False,
    ):
        self.weighted_tfidf = FieldWeightedTFIDFVectorizer(
            n_features,
            field_weights,
            description_truncate_chars,
            ngram_range=ngram_range,
            sublinear_tf=sublinear_tf,
        )
        self.svd = TruncatedSVD(n_components=n_components, random_state=CONFIG["model_seed"])
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None

    def fit_transform(self, items_df: pd.DataFrame) -> np.ndarray:
        weighted_matrix = self.weighted_tfidf.fit_transform(items_df)  # sparse, already L2-normalised
        self.item_idx_order = self.weighted_tfidf.item_idx_order
        reduced = self.svd.fit_transform(weighted_matrix).astype(np.float32)
        self.item_matrix = normalize(reduced)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        n_neighbors = min(int(n_neighbors), int(self.item_matrix.shape[0]))
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors: np.ndarray, k: int):
        k = min(int(k), int(self.item_matrix.shape[0]))
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, new_items: pd.DataFrame):
        weighted_counts = self.weighted_tfidf.transform_new_items(new_items)
        reduced = self.svd.transform(weighted_counts).astype(np.float32)
        return normalize(reduced)


### 5.8 User Profile Construction
User profiles are constructed from the content vectors of the books with which each user has previously interacted. Each item vector is weighted by the user's normalised rating. A configurable multiplier can also be applied to verified purchases, allowing these interactions to contribute more strongly to the user profile.

The weighted item vectors are summed and L2-normalised to produce a single content-preference vector for each user. The interacted-item sets are also stored so that previously seen books can be excluded from the recommendation results. Two profile-construction strategies are provided.

`LazyUserProfiles` computes a user's profile only when requested; It is suitable for high-dimensional, sparse TF-IDF representations.

In contrast, `build_user_profiles_dense()` computes all user profiles at once through sparse matrix multiplication and is used for lower-dimensional dense representations, such as the vectors produced by SVD.


In [83]:
class LazyUserProfiles:
    def __init__(
        self,
        interactions_df,
        item_matrix,
        item_idx_to_row,
        verified_purchase_multiplier: float,
    ):
        df = interactions_df.copy()
        df["row"] = df["item_idx"].map(item_idx_to_row)
        df = df.dropna(subset=["row"])
        df["row"] = df["row"].astype(int)

        rating_weight = df["rating"].to_numpy() / CONFIG["rating_scale_max"]
        verified_mult = np.where(
            df["verified_purchase"].to_numpy(dtype=bool),
            verified_purchase_multiplier,
            1.0,
        )
        df["weight"] = rating_weight * verified_mult

        self.user_rows = df.groupby("user_idx")["row"].apply(list).to_dict()
        self.user_weights = df.groupby("user_idx")["weight"].apply(list).to_dict()
        self.interacted_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()
        self.item_matrix = item_matrix
        self.vocab_size = item_matrix.shape[1]
        self.is_sparse = sp.issparse(item_matrix)

    def __getitem__(self, user_idx: int):
        rows = self.user_rows.get(user_idx)
        if not rows:
            if self.is_sparse:
                return sp.csr_matrix((1, self.vocab_size))
            return np.zeros((1, self.vocab_size), dtype=np.float32)

        weights = np.asarray(self.user_weights[user_idx])
        vecs = self.item_matrix[rows]
        if self.is_sparse:
            weighted = sp.diags(weights) @ vecs
            profile = sp.csr_matrix(weighted.sum(axis=0))
            norm = np.sqrt(profile.multiply(profile).sum())
            if norm > 0:
                profile = profile / norm
            return profile

        profile = (vecs * weights[:, None]).sum(axis=0, keepdims=True)
        norm = np.linalg.norm(profile)
        if norm > 0:
            profile = profile / norm
        return profile.astype(np.float32)


def build_user_profiles_dense(
    interactions_df,
    item_matrix,
    n_users,
    item_idx_to_row,
    verified_purchase_multiplier: float,
):
    df = interactions_df.copy()
    df["row"] = df["item_idx"].map(item_idx_to_row)
    df = df.dropna(subset=["row"])
    df["row"] = df["row"].astype(int)

    rating_weight = df["rating"].to_numpy() / CONFIG["rating_scale_max"]
    verified_mult = np.where(
        df["verified_purchase"].to_numpy(dtype=bool),
        verified_purchase_multiplier,
        1.0,
    )
    weight = rating_weight * verified_mult

    n_matrix_items = item_matrix.shape[0]
    interaction_weights = sp.coo_matrix(
        (weight, (df["user_idx"].to_numpy(), df["row"].to_numpy())),
        shape=(n_users, n_matrix_items),
    ).tocsr()
    profiles = interaction_weights @ item_matrix

    norms = np.linalg.norm(profiles, axis=1)
    norms[norms == 0] = 1.0
    profiles = (profiles.T / norms).T.astype(np.float32)
    interacted_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    class DenseProfileStore:
        is_sparse = False

        def __init__(self, matrix):
            self._matrix = matrix

        def __getitem__(self, index):
            return self._matrix[index].reshape(1, -1)

    return DenseProfileStore(profiles), interacted_items


# 5.9 Content-Based Recommendation and Result Presentation

`ContentRecommender` generates the Top-K recommendation list by matching the
user profile with item content vectors, filtering previously interacted items,
and applying a popularity fallback when necessary. Its `score_candidates()`
method is used for offline ranking evaluation.


In [84]:
class ContentRecommender:
    def __init__(self, vectorizer, items_df, profiles, interacted_items, item_idx_to_row):
        self.vectorizer = vectorizer
        self.items_df = items_df.set_index("item_idx")
        self.profiles = profiles
        self.interacted_items = interacted_items
        self.item_idx_to_row = item_idx_to_row
        self.row_to_item_idx = vectorizer.item_idx_order
        self.is_sparse = getattr(profiles, "is_sparse", None)
        if self.is_sparse is None:
            self.is_sparse = sp.issparse(profiles)
        self.popularity_rank = items_df.sort_values("popularity_score", ascending=False)["item_idx"].tolist()
        self.cold_start_builder = None
        self.pending_new_items = {}

    def _get_profile_vec(self, user_idx: int):
        vec = self.profiles[user_idx]
        if self.is_sparse:
            return vec, vec.nnz == 0
        vec = vec.reshape(1, -1)
        return vec, not np.any(vec)

    def _popularity_fallback(self, exclude: set, k: int):
        recs = []
        for item_idx in self.popularity_rank:
            if item_idx in exclude:
                continue
            recs.append(item_idx)
            if len(recs) == k:
                break
        return recs

    def _mmr_select(
        self,
        candidates: list[tuple[int, float]],
        k: int,
        relevance_weight: float = 0.85,
    ) -> list[tuple[int, float]]:
        if len(candidates) <= k:
            return candidates
        scores = np.array([s for _, s in candidates], dtype=np.float64)
        score_range = max(scores.max() - scores.min(), 1e-12)
        relevance = (scores - scores.min()) / score_range

        selected: list[int] = []
        remaining = list(range(len(candidates)))
        while len(selected) < k and remaining:
            def mmr_value(position):
                redundancy = max(
                    (
                        item_metadata_similarity(self.items_df, candidates[position][0], candidates[chosen][0])
                        for chosen in selected
                    ),
                    default=0.0,
                )
                return relevance_weight * relevance[position] - (1 - relevance_weight) * redundancy

            best = max(remaining, key=mmr_value)
            selected.append(best)
            remaining.remove(best)
        return [candidates[i] for i in selected]

    def recommend(
        self,
        user_idx: int,
        k: int = None,
        candidate_pool: int = None,
        diversify: bool = True,
        mmr_relevance_weight: float = 0.85,
    ):
        k = k or CONFIG["top_k"]
        candidate_pool = candidate_pool or CONFIG["candidate_pool"]
        seen = self.interacted_items.get(user_idx, set())
        profile_vec, is_cold = self._get_profile_vec(user_idx)

        if is_cold:
            item_ids = self._popularity_fallback(seen, k)
            return [{"item_idx": i, "reason": "popular", "similarity": 0.0,
                      "title": self.items_df.loc[i, "title"]} for i in item_ids]

        dist, idx = self.vectorizer.query_topk(profile_vec, k=candidate_pool)
        dist, idx = dist[0], idx[0]
        sim = 1 - dist

        candidates = []
        for row, s in zip(idx, sim):
            item_idx = int(self.row_to_item_idx[row])
            if item_idx in seen:
                continue
            candidates.append((item_idx, float(s)))

        for item_idx, new_vector in self.pending_new_items.items():
            if item_idx in seen:
                continue
            candidates.append((item_idx, self._new_item_similarity(new_vector, profile_vec)))

        if diversify and len(candidates) > k:
            results = self._mmr_select(candidates, k, mmr_relevance_weight)
        else:
            results = candidates[:k]

        if len(results) < k:
            fill = self._popularity_fallback(seen | {r[0] for r in results}, k - len(results))
            results += [(i, 0.0) for i in fill]

        return [{"item_idx": i, "reason": "content-similar" if s > 0 else "popular",
                  "similarity": round(float(s), 4), "title": self.items_df.loc[i, "title"]}
                 for i, s in results]

    def score_candidates(self, user_idx: int, candidate_item_idxs: list) -> np.ndarray:
        profile_vec, is_cold = self._get_profile_vec(user_idx)
        if is_cold:
            return self.items_df.loc[candidate_item_idxs, "popularity_score"].to_numpy()
        rows = [self.item_idx_to_row[i] for i in candidate_item_idxs]
        cand_matrix = self.vectorizer.item_matrix[rows]
        if self.is_sparse:
            return (cand_matrix @ profile_vec.T).toarray().ravel()
        return cand_matrix @ profile_vec.ravel()

    def attach_cold_start(self, builder: "ColdStartProfileBuilder", diversify_fallback: bool = True) -> None:
        self.cold_start_builder = builder
        if diversify_fallback:
            self.popularity_rank = builder.diversified_popularity_order()

    def recommend_for_new_user(self, preferred_categories: list[str] | None = None, k: int | None = None):
        k = k or CONFIG["top_k"]
        if preferred_categories and self.cold_start_builder is not None:
            profile_vec = self.cold_start_builder.onboarding_profile(preferred_categories)
            if profile_vec is not None:
                candidate_pool = CONFIG["candidate_pool"]
                dist, idx = self.vectorizer.query_topk(profile_vec, k=candidate_pool)
                dist, idx = dist[0], idx[0]
                sim = 1 - dist
                results = []
                for row, s in zip(idx, sim):
                    item_idx = int(self.row_to_item_idx[row])
                    results.append((item_idx, s))
                    if len(results) == k:
                        break
                return [
                    {"item_idx": i, "reason": "onboarding-categories", "similarity": round(float(s), 4),
                     "title": self.items_df.loc[i, "title"]}
                    for i, s in results
                ]
        fallback = (
            self.cold_start_builder.diversified_popularity_order()
            if self.cold_start_builder is not None
            else self.popularity_rank
        )
        return [
            {"item_idx": i, "reason": "popular", "similarity": 0.0, "title": self.items_df.loc[i, "title"]}
            for i in fallback[:k]
        ]

    def _new_item_input(self, item_fields: dict[str, str]):
        one_row = pd.DataFrame([
            {field: item_fields.get(field, "") for field in ITEM_TEXT_FIELDS}
        ])
        if isinstance(self.vectorizer, (FieldWeightedTFIDFVectorizer, FieldWeightedSVDContentVectorizer)):
            return one_row
        content_text = build_unweighted_content_text(one_row, CONFIG["description_truncate_chars"]).tolist()
        return content_text

    def score_new_item(self, item_fields: dict[str, str]) -> np.ndarray:
        return self.vectorizer.transform_new_items(self._new_item_input(item_fields))

    def _new_item_similarity(self, new_vector, profile_vec) -> float:
        if self.is_sparse:
            new_vector = new_vector if sp.issparse(new_vector) else sp.csr_matrix(new_vector)
            return float((new_vector @ profile_vec.T).toarray().ravel()[0])
        return float(np.asarray(new_vector).ravel() @ np.asarray(profile_vec).ravel())

    def register_new_item(
        self,
        item_idx: int,
        item_fields: dict[str, str],
        title: str | None = None,
    ) -> np.ndarray:
        if item_idx in self.item_idx_to_row:
            raise ValueError(f"item_idx={item_idx} already exists in the fitted item index")
        vector = self.score_new_item(item_fields)
        self.pending_new_items[item_idx] = vector
        self.items_df.loc[item_idx, "title"] = title or item_fields.get("title", "")
        self.items_df.loc[item_idx, "categories"] = item_fields.get("categories", "")
        self.items_df.loc[item_idx, "author"] = item_fields.get("author", "")
        return vector


`recommend_as_dataframe()` does not calculate new recommendation scores. It
formats the output of `recommend()` as a DataFrame and adds ranking positions
and item metadata for clearer inspection and presentation.

In [85]:
def recommend_as_dataframe(
    recommender,
    user_idx: int,
    k: int | None = None,
    **recommend_kwargs,
) -> pd.DataFrame:
    k = k or CONFIG["top_k"]
    recommendations = recommender.recommend(user_idx, k=k, **recommend_kwargs)
    frame = pd.DataFrame(recommendations)
    if frame.empty:
        columns = ["rank", "item_idx", "score", "title", "reason"]
        return pd.DataFrame(columns=columns)

    frame = frame.rename(columns={"similarity": "score"})
    frame.insert(0, "rank", np.arange(1, len(frame) + 1))

    metadata_cols = [
        col for col in ["author", "categories", "average_rating", "rating_number"]
        if col in recommender.items_df.columns
    ]
    if metadata_cols:
        frame = frame.merge(
            recommender.items_df[metadata_cols],
            left_on="item_idx",
            right_index=True,
            how="left",
        )

    ordered_cols = ["rank", "item_idx", "score", "title", *metadata_cols, "reason"]
    return frame[[col for col in ordered_cols if col in frame.columns]]


### 5.10 Evaluation

Each model is evaluated by ranking one held-out positive item against 100 sampled negatives. The positive-item ranks are summarised using Hit@K, Recall@K, Precision@K, NDCG@K, and MRR. Candidate order is shuffled with a fixed evaluation seed to ensure reproducible comparisons.

The final Top-K lists are also assessed using novelty, intra-list diversity, average training popularity, and catalogue coverage. Candidate scores may be standardised for later hybrid fusion, and the latest evaluation result for each model and split is saved to a shared CSV file.


In [86]:
def rank_from_scores(
    user_idx: int,
    candidates: list[int],
    scores: np.ndarray,
    seed: int,
) -> int:
    candidates = np.asarray(candidates, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if len(candidates) != len(scores):
        raise ValueError("Candidate and score lengths differ")

    # The positive starts at candidate position zero.
    rng = np.random.default_rng(seed + int(user_idx))
    permutation = rng.permutation(len(candidates))
    shuffled_scores = scores[permutation]
    shuffled_positive_position = int(np.flatnonzero(permutation == 0)[0])
    order = np.argsort(-shuffled_scores, kind="stable")
    return int(np.flatnonzero(order == shuffled_positive_position)[0]) + 1


def summarise_ranks(
    ranks: list[int] | np.ndarray,
    k: int,
    model_name: str,
    split: str,
    num_candidates: int = 101,
) -> dict:
    ranks = np.asarray(ranks, dtype=np.int64)
    if not 0 < k < num_candidates:
        raise ValueError("k must be between 1 and num_candidates - 1")
    hits = ranks <= k
    hit_rate = float(hits.mean())
    return {
        "model": model_name,
        "split": normalise_split_name(split),
        f"Hit@{k}": hit_rate,
        f"Recall@{k}": hit_rate,
        f"Precision@{k}": hit_rate / k,
        f"CandidateAccuracy@{k}": float((num_candidates - k + hit_rate) / num_candidates),
        f"NDCG@{k}": float(
            np.where(hits, 1.0 / np.log2(ranks + 1), 0.0).mean()
        ),
        "MRR": float((1.0 / ranks).mean()),
        "users": int(len(ranks)),
    }


def evaluate_ranking(
    recommender,
    positives_df,
    negatives_dict,
    k: int,
    model_name: str,
    split: str,
    seed: int | None = None,
) -> dict:
    seed = CONFIG["evaluation_seed"] if seed is None else seed
    ranks = []
    num_candidates = 101
    for row in positives_df.itertuples():
        user_idx = int(row.user_idx)
        positive = int(row.pos_idx)
        candidates = [positive, *negatives_dict[user_idx]]
        num_candidates = len(candidates)
        scores = recommender.score_candidates(user_idx, candidates)
        ranks.append(rank_from_scores(user_idx, candidates, scores, seed))
    return summarise_ranks(ranks, k, model_name, split, num_candidates=num_candidates)


def item_metadata_similarity(items_indexed: pd.DataFrame, left: int, right: int) -> float:
    left_categories = {
        value.strip()
        for value in str(items_indexed.at[left, "categories"]).split(";")
        if value.strip()
    }
    right_categories = {
        value.strip()
        for value in str(items_indexed.at[right, "categories"]).split(";")
        if value.strip()
    }
    category_similarity = len(left_categories & right_categories) / max(
        len(left_categories | right_categories), 1
    )
    left_author = str(items_indexed.at[left, "author"]).strip()
    right_author = str(items_indexed.at[right, "author"]).strip()
    author_similarity = float(bool(left_author) and left_author == right_author)
    return max(category_similarity, author_similarity)


def evaluate_catalogue_diagnostics(
    recommender,
    user_idxs,
    n_total_items: int,
    k: int,
    train_item_counts: np.ndarray,
) -> tuple[pd.DataFrame, dict]:
    recommended_items = set()
    total_train_interactions = int(train_item_counts.sum())
    rows = []

    for user_idx in user_idxs:
        recommendations = recommender.recommend(int(user_idx), k=k)
        items = [int(row["item_idx"]) for row in recommendations]
        recommended_items.update(items)

        novelty = float(np.mean([
            -np.log2(
                (train_item_counts[item] + 1) / (total_train_interactions + n_total_items)
            )
            for item in items
        ]))
        pair_similarities = [
            item_metadata_similarity(recommender.items_df, left, right)
            for position, left in enumerate(items)
            for right in items[position + 1 :]
        ]
        intra_list_diversity = (
            1.0 - float(np.mean(pair_similarities)) if pair_similarities else 0.0
        )
        avg_train_popularity = float(np.mean(train_item_counts[items]))

        rows.append({
            "user_idx": int(user_idx),
            "novelty": novelty,
            "intra_list_diversity": intra_list_diversity,
            "avg_train_popularity": avg_train_popularity,
        })

    diagnostics_df = pd.DataFrame(rows)
    catalogue_metrics = {
        "users": len(user_idxs),
        "catalogue_coverage": len(recommended_items) / n_total_items,
        "unique_recommended_items": len(recommended_items),
    }
    return diagnostics_df, catalogue_metrics


def standardise_candidate_scores(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64)
    std = scores.std()
    return (scores - scores.mean()) / max(std, 1e-8)

# SHARED_OUTPUTS_DIR = (
#     Path(COLAB_OUTPUT_ROOT) / "outputs" if IN_COLAB else Path("outputs")
# )
SHARED_OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_candidate_set_results(
    rows_df: pd.DataFrame,
    path: Path = SHARED_OUTPUTS_DIR / "candidate_set_results.csv",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, rows_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=["model", "split"], keep="last")
    else:
        combined = rows_df
    combined.to_csv(path, index=False)


In [87]:
REPORT_NAME_MAP = {
    "Popularity": "Popularity (no personalisation)",
    "Basic-TFIDF": "TFIDF-Cosine",
    "Weighted-TFIDF": "Field-Weighted TFIDF-Cosine",
    "ContentBased-SVD": "TFIDF-SVD-Cosine",
}

def to_report_names(df, name_col="model"):
    df = df.copy()
    df[name_col] = df[name_col].map(lambda n: REPORT_NAME_MAP.get(n, n))
    return df

In [88]:
def build_recommender_for_history(
    vectorizer,
    items_df,
    interactions_df,
    n_users,
    item_idx_to_row,
    use_lazy_profiles,
    verified_purchase_multiplier,
):
    if use_lazy_profiles:
        profiles = LazyUserProfiles(
            interactions_df,
            vectorizer.item_matrix,
            item_idx_to_row,
            verified_purchase_multiplier,
        )
        interacted = profiles.interacted_items
    else:
        profiles, interacted = build_user_profiles_dense(
            interactions_df,
            vectorizer.item_matrix,
            n_users,
            item_idx_to_row,
            verified_purchase_multiplier,
        )
    return ContentRecommender(
        vectorizer,
        items_df,
        profiles,
        interacted,
        item_idx_to_row,
    )


def profile_history_for_split(split: str) -> pd.DataFrame:
    split = normalise_split_name(split)
    if split == "validation":
        return train_interactions
    if split == "test":
        return test_profile_interactions
    raise ValueError("Evaluation is defined only for validation or test")


def run_variant(
    name,
    vectorizer,
    items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles,
    eval_splits=("validation",),
    verified_purchase_multiplier: float | None = None,
):
    del train_interactions, n_items
    verified_purchase_multiplier = (
        CONFIG["verified_purchase_multiplier"]
        if verified_purchase_multiplier is None
        else verified_purchase_multiplier
    )
    eval_splits = tuple(normalise_split_name(split) for split in eval_splits)

    progress(f"[{name}] fitting item content representation...")
    item_matrix = vectorizer.fit_transform(items_df)
    item_idx_to_row = {
        int(item_idx): row
        for row, item_idx in enumerate(vectorizer.item_idx_order)
    }
    progress(f"[{name}] item_matrix shape={item_matrix.shape}", done=True)
    vectorizer.build_index(n_neighbors=CONFIG["n_neighbors_index"])

    results = []
    recommenders_by_split = {}
    for split in eval_splits:
        history = profile_history_for_split(split)
        recommender = build_recommender_for_history(
            vectorizer,
            items_df,
            history,
            n_users,
            item_idx_to_row,
            use_lazy_profiles,
            verified_purchase_multiplier,
        )
        recommenders_by_split[split] = recommender
        positives = load_eval_positives(DATA_DIR, split)
        negatives = load_eval_negatives(DATA_DIR, split)
        metrics = evaluate_ranking(
            recommender,
            positives,
            negatives,
            CONFIG["eval_k"],
            name,
            split,
        )
        results.append(metrics)
        progress(
            f"[{name}] {split}: "
            f"Hit@10={metrics['Hit@10']:.4f} "
            f"NDCG@10={metrics['NDCG@10']:.4f} "
            f"MRR={metrics['MRR']:.4f}",
            done=True,
        )

    return recommenders_by_split[eval_splits[-1]], results


def run_all_variants(
    items_df,
    train_interactions,
    n_users,
    n_items,
    eval_splits=("validation",),
) -> dict[str, Any]:
    started = time.time()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    eval_splits = tuple(normalise_split_name(split) for split in eval_splits)

    diagnostic_rng = np.random.default_rng(CONFIG["evaluation_seed"])
    diagnostic_users = diagnostic_rng.choice(
        np.arange(n_users),
        size=min(CONFIG["diagnostic_n_users"], n_users),
        replace=False,
    )

    all_results = []
    catalogue_results = []
    catalogue_diagnostics_by_model = {}
    recommenders = {}

    def register(name, recommender, results):
        recommenders[name] = recommender
        all_results.extend(results)
        if "test" in eval_splits:
            per_user_diagnostics, diagnostics = evaluate_catalogue_diagnostics(
                recommender,
                diagnostic_users,
                n_items,
                CONFIG["top_k"],
                train_item_counts,
            )
            diagnostics["model"] = name
            catalogue_results.append(diagnostics)
            catalogue_diagnostics_by_model[name] = per_user_diagnostics

    popularity_history = (
        test_interacted_items if "test" in eval_splits else train_interacted_items
    )
    popularity_recommender = PopularityRecommender(items_df, popularity_history)
    popularity_results = []
    for split in eval_splits:
        metrics = evaluate_ranking(
            popularity_recommender,
            load_eval_positives(DATA_DIR, split),
            load_eval_negatives(DATA_DIR, split),
            CONFIG["eval_k"],
            "Popularity",
            split,
        )
        popularity_results.append(metrics)
        progress(
            f"[Popularity] {split}: Hit@10={metrics['Hit@10']:.4f} "
            f"NDCG@10={metrics['NDCG@10']:.4f} MRR={metrics['MRR']:.4f}",
            done=True,
        )
    register("Popularity", popularity_recommender, popularity_results)

    variant_specs = [
        (
            "Basic-TFIDF",
            TFIDFHashingVectorizer(CONFIG["basic_n_features"]),
            True,
        ),
        (
            "ContentBased-SVD",
            SVDContentVectorizer(
                CONFIG["svd_max_features"],
                CONFIG["svd_min_df"],
                CONFIG["svd_n_components"],
            ),
            False,
        ),
    ]

    for name, vectorizer, use_lazy_profiles in variant_specs:
        recommender, results = run_variant(
            name,
            vectorizer,
            items_df,
            train_interactions,
            n_users,
            n_items,
            use_lazy_profiles,
            eval_splits=eval_splits,
        )
        register(name, recommender, results)

    results_df = pd.DataFrame(all_results)
    catalogue_df = pd.DataFrame(catalogue_results)

    save_candidate_set_results(to_report_names(results_df))

    with open(
        OUTPUT_DIR / "validation_model_comparison.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            {
                "protocol": {
                    "validation_context": "train",
                    "test_context": "train+validation",
                    "primary_metric": CONFIG["primary_metric"],
                    "evaluation_seed": CONFIG["evaluation_seed"],
                },
                "results_table": all_results,
                "elapsed_seconds": time.time() - started,
            },
            handle,
            indent=2,
        )
    progress(f"Validation comparison done in {time.time() - started:.1f}s", done=True)
    return recommenders, results_df, catalogue_df


In [89]:
recommenders, results_df, _ = run_all_variants(
    items_df,
    train_interactions,
    n_users,
    n_items,
    eval_splits=("validation",),
)

print(
    "Validation-only model comparison. "
    f"Primary selection metric: {CONFIG['primary_metric']}"
)
display(
    to_report_names(results_df).sort_values(
        CONFIG["primary_metric"],
        ascending=False,
    )
)


[Popularity] validation: Hit@10=0.4056 NDCG@10=0.2533 MRR=0.2248                                    
[Basic-TFIDF] item_matrix shape=(81322, 32768)                                                      
[Basic-TFIDF] validation: Hit@10=0.5375 NDCG@10=0.3899 MRR=0.3614                                   
[ContentBased-SVD] item_matrix shape=(81322, 128)                                                   
[ContentBased-SVD] validation: Hit@10=0.5831 NDCG@10=0.3806 MRR=0.3344                              
Validation comparison done in 92.8s                                                                 
Validation-only model comparison. Primary selection metric: NDCG@10


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
1,TFIDF-Cosine,validation,0.537484,0.537484,0.053748,0.906312,0.389874,0.361362,64867
2,TFIDF-SVD-Cosine,validation,0.583085,0.583085,0.058309,0.906763,0.380551,0.334444,64867
0,Popularity (no personalisation),validation,0.405645,0.405645,0.040565,0.905006,0.253269,0.224836,64867


### Sample recommendations per variant

In [90]:
for name, rec in recommenders.items():
    print(f"=== {name} ===")
    for u in [0, 1]:
        print(f"  user {u}:")
        for r in rec.recommend(u, k=3):
            print(f"    - [{r['reason']}] sim={r['similarity']:.3f} {r['title'][:55]}")

=== Popularity ===
  user 0:
    - [popular] sim=0.000 My Sister's Grave (Tracy Crosswhite Book 1)
    - [popular] sim=0.000 Orphan Train: A Novel
    - [popular] sim=0.000 Where the Crawdads Sing
  user 1:
    - [popular] sim=0.000 My Sister's Grave (Tracy Crosswhite Book 1)
    - [popular] sim=0.000 Orphan Train: A Novel
    - [popular] sim=0.000 Where the Crawdads Sing
=== Basic-TFIDF ===
  user 0:
    - [content-similar] sim=0.242 Atomic Habits: An Easy & Proven Way to Build Good Habit
    - [content-similar] sim=0.201 Turtles All the Way Down
    - [content-similar] sim=0.212 The Undoing Project: A Friendship That Changed Our Mind
  user 1:
    - [content-similar] sim=0.316 Deadly Associations (Book #3 in The Claudia Hershey Mys
    - [content-similar] sim=0.312 Quinn Goes to Jail (Liam Quinn Mysteries Book 8)
    - [content-similar] sim=0.286 Margaritas & Murder: A Sunny Truly Mystery (Sunny Truly
=== ContentBased-SVD ===
  user 0:
    - [content-similar] sim=0.900 Thank You for 

## 5.2 hyperparameter Tuning

### 5.21 hyperparameter tuning - for SVD dimension
The SVD dimension is tuned on the validation split by comparing 32, 64, 128,
and 256 components. The candidate with the highest validation value of the
primary metric is selected as `BEST_SVD_DIM` and used in the final SVD-based
model.

In [91]:
svd_dim_results = []
for dim in [32, 64, 128, 256]:
    svd_vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        dim,
    )
    _, result = run_variant(
        f"SVD-{dim}dim",
        svd_vectorizer,
        items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
    )
    row = result[0]
    row["svd_dim"] = dim
    svd_dim_results.append(row)

svd_dim_df = pd.DataFrame(svd_dim_results)
best_svd_row = svd_dim_df.loc[
    svd_dim_df[CONFIG["primary_metric"]].idxmax()
]
BEST_SVD_DIM = int(best_svd_row["svd_dim"])
CONFIG["svd_n_components"] = BEST_SVD_DIM

print(
    f"Selected SVD dimension={BEST_SVD_DIM} using "
    f"validation {CONFIG['primary_metric']}="
    f"{best_svd_row[CONFIG['primary_metric']]:.6f}"
)
display(svd_dim_df.sort_values(CONFIG["primary_metric"], ascending=False))


[SVD-32dim] item_matrix shape=(81322, 32)                                                           
[SVD-32dim] validation: Hit@10=0.5749 NDCG@10=0.3544 MRR=0.3042                                     
[SVD-64dim] item_matrix shape=(81322, 64)                                                           
[SVD-64dim] validation: Hit@10=0.5835 NDCG@10=0.3722 MRR=0.3239                                     
[SVD-128dim] item_matrix shape=(81322, 128)                                                         
[SVD-128dim] validation: Hit@10=0.5831 NDCG@10=0.3806 MRR=0.3344                                    
[SVD-256dim] item_matrix shape=(81322, 256)                                                         
[SVD-256dim] validation: Hit@10=0.5745 NDCG@10=0.3857 MRR=0.3438                                    
Selected SVD dimension=256 using validation NDCG@10=0.385707


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users,svd_dim
3,SVD-256dim,validation,0.574514,0.574514,0.057451,0.906678,0.385707,0.343840,64867,256
2,SVD-128dim,validation,0.583085,0.583085,0.058309,0.906763,0.380551,0.334444,64867,128
1,SVD-64dim,validation,0.583486,0.583486,0.058349,0.906767,0.372167,0.323866,64867,64
0,SVD-32dim,validation,0.574930,0.574930,0.057493,0.906682,0.354405,0.304214,64867,32


In [92]:
assert CONFIG["svd_n_components"] == BEST_SVD_DIM
print(
    f"Locked SVD dimension from validation: {BEST_SVD_DIM}. "
    "Test remains unevaluated in this revised workflow."
)


Locked SVD dimension from validation: 256. Test remains unevaluated in this revised workflow.


### 5.22 Author-Field Ablation

By comparing the four candidate dimensions (32, 64, 128, and 256), the SVD dimension is tuned on the validation set. Finally, the dimension with the BEST performance of the main evaluation index was selected and recorded as BEST SVD DIM for constructing the final SVD content recommendation model.

In [93]:
def build_content_text(items_df, description_truncate_chars, include_author=False):
    truncated_description = items_df["description"].str.slice(0, description_truncate_chars)
    content = (
        (items_df["title"] + " ") * 2
        + (items_df["categories"].str.replace(";", " ", regex=False) + " ") * 2
    )
    if include_author:
        content = content + items_df["author"] + " "
    content = content + items_df["features"] + " " + truncated_description
    return content.str.strip()


items_df_no_author = items_df.copy()
items_df_with_author = items_df.copy()
items_df_with_author["content_text"] = build_content_text(
    items_df_with_author,
    CONFIG["description_truncate_chars"],
    include_author=True,
)

author_ablation_results = []
for label, frame in [
    ("no-author", items_df_no_author),
    ("with-author", items_df_with_author),
]:
    vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    )
    _, result = run_variant(
        f"SVD-{label}",
        vectorizer,
        frame,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
    )
    result[0]["variant"] = label
    author_ablation_results.append(result[0])

author_ablation_df = pd.DataFrame(author_ablation_results)
best_author_row = author_ablation_df.loc[
    author_ablation_df[CONFIG["primary_metric"]].idxmax()
]
BEST_INCLUDE_AUTHOR = best_author_row["variant"] == "with-author"

print(
    f"Selected include_author={BEST_INCLUDE_AUTHOR} using "
    f"validation {CONFIG['primary_metric']}="
    f"{best_author_row[CONFIG['primary_metric']]:.6f}"
)
display(
    author_ablation_df[
        ["variant", "Hit@10", "NDCG@10", "MRR"]
    ].sort_values(CONFIG["primary_metric"], ascending=False)
)


[SVD-no-author] item_matrix shape=(81322, 256)                                                      
[SVD-no-author] validation: Hit@10=0.5745 NDCG@10=0.3857 MRR=0.3438                                 
[SVD-with-author] item_matrix shape=(81322, 256)                                                    
[SVD-with-author] validation: Hit@10=0.5895 NDCG@10=0.3974 MRR=0.3541                               
Selected include_author=True using validation NDCG@10=0.397368


,variant,Hit@10,NDCG@10,MRR
1,with-author,0.589499,0.397368,0.354133
0,no-author,0.574514,0.385707,0.343840


### 5.23 Candidate-Pool Size Diagnostic
This diagnostic experiment is used to examine the size of the retrieval candidate pool, thereby generating a complete list of Top-10 content recommendations.

In [94]:
def measure_fallback_rate(recommender, user_idxs, k=10, candidate_pool=50):
    total_popular, total_recs, users_with_fallback = 0, 0, 0
    for user in user_idxs:
        recommendations = recommender.recommend(
            int(user),
            k=k,
            candidate_pool=candidate_pool,
        )
        n_popular = sum(
            row["reason"] == "popular"
            for row in recommendations
        )
        total_popular += n_popular
        total_recs += len(recommendations)
        users_with_fallback += int(n_popular > 0)
    return {
        "candidate_pool": candidate_pool,
        "avg_popular_items_per_user": total_popular / len(user_idxs),
        "pct_users_with_any_fallback": users_with_fallback / len(user_idxs),
        "pct_recs_that_are_fallback": total_popular / total_recs,
    }


diagnostic_rng = np.random.default_rng(CONFIG["evaluation_seed"])
sample_users_for_pool_test = diagnostic_rng.choice(
    np.arange(n_users),
    size=min(CONFIG["diagnostic_n_users"], n_users),
    replace=False,
)

pool_ablation_results = []
for pool in [5, 10, 20, 50]:
    stats = measure_fallback_rate(
        recommenders["ContentBased-SVD"],
        sample_users_for_pool_test,
        k=10,
        candidate_pool=pool,
    )
    pool_ablation_results.append(stats)

pool_ablation_df = pd.DataFrame(pool_ablation_results)
display(pool_ablation_df)


,candidate_pool,avg_popular_items_per_user,pct_users_with_any_fallback,pct_recs_that_are_fallback
0,5,6.1998,1.0000,0.61998
1,10,1.5718,0.7684,0.15718
2,20,0.0066,0.0020,0.00066
3,50,0.0000,0.0000,0.00000


### 5.24 Verified-Purchase Weight Tuning

Compare the four verified purchase weights (1.0, 1.2, 1.5, and 2.0) on the validation set. This weight controls the importance of verified purchase interactions in the user profile, based on rating weighting. Ultimately, the weight of the highest validation set result on the main evaluation metric is selected and saved as the BEST VERIFIED MULTIPLIER for the final content recommendation model.

In [95]:
verified_items_df = items_df.copy()
verified_items_df["content_text"] = build_content_text(
    verified_items_df,
    CONFIG["description_truncate_chars"],
    include_author=BEST_INCLUDE_AUTHOR,
)

verified_multiplier_results = []
for multiplier in [1.0, 1.2, 1.5, 2.0]:
    vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    )
    _, result = run_variant(
        f"SVD-verified{multiplier}",
        vectorizer,
        verified_items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
        verified_purchase_multiplier=multiplier,
    )
    result[0]["verified_multiplier"] = multiplier
    verified_multiplier_results.append(result[0])

verified_multiplier_df = pd.DataFrame(verified_multiplier_results)
best_verified_row = verified_multiplier_df.loc[
    verified_multiplier_df[CONFIG["primary_metric"]].idxmax()
]
BEST_VERIFIED_MULTIPLIER = float(
    best_verified_row["verified_multiplier"]
)
CONFIG["verified_purchase_multiplier"] = BEST_VERIFIED_MULTIPLIER

print(
    f"Selected verified_purchase_multiplier={BEST_VERIFIED_MULTIPLIER} "
    f"using validation {CONFIG['primary_metric']}="
    f"{best_verified_row[CONFIG['primary_metric']]:.6f}"
)
display(
    verified_multiplier_df[
        ["verified_multiplier", "Hit@10", "NDCG@10", "MRR"]
    ].sort_values(CONFIG["primary_metric"], ascending=False)
)


[SVD-verified1.0] item_matrix shape=(81322, 256)                                                    
[SVD-verified1.0] validation: Hit@10=0.5895 NDCG@10=0.3974 MRR=0.3541                               
[SVD-verified1.2] item_matrix shape=(81322, 256)                                                    
[SVD-verified1.2] validation: Hit@10=0.5890 NDCG@10=0.3968 MRR=0.3535                               
[SVD-verified1.5] item_matrix shape=(81322, 256)                                                    
[SVD-verified1.5] validation: Hit@10=0.5877 NDCG@10=0.3956 MRR=0.3524                               
[SVD-verified2.0] item_matrix shape=(81322, 256)                                                    
[SVD-verified2.0] validation: Hit@10=0.5857 NDCG@10=0.3935 MRR=0.3504                               
Selected verified_purchase_multiplier=1.0 using validation NDCG@10=0.397368


,verified_multiplier,Hit@10,NDCG@10,MRR
0,1.0,0.589499,0.397368,0.354133
1,1.2,0.589005,0.396770,0.353506
2,1.5,0.587726,0.395571,0.352383
3,2.0,0.585675,0.393538,0.350428


### 5.25 Field-Weight Tuning for Weighted TF-IDF

While keeping other model Settings unchanged, the code will compare different combinations of field weights on the validation set. Each configuration will set different weights for the available book metadata fields. Finally, select the configuration that achieves the highest validation set score on the main evaluation metric and save it as `BEST WEIGHTED FIELD WEIGHTS`.


In [96]:
weighted_tfidf_tuning_rows = []
for label, field_weights in CONFIG["weighted_field_weight_candidates"].items():
    _, result = run_variant(
        f"Weighted-TFIDF-{label}",
        FieldWeightedTFIDFVectorizer(
            CONFIG["basic_n_features"],
            field_weights,
            CONFIG["description_truncate_chars"],
            ngram_range=CONFIG["weighted_tfidf_ngram_range"],
            sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
        ),
        items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=True,
        eval_splits=("validation",),
    )
    row = result[0]
    row["weight_label"] = label
    row["field_weights"] = json.dumps(field_weights, sort_keys=True)
    weighted_tfidf_tuning_rows.append(row)

weighted_tfidf_tuning_df = pd.DataFrame(weighted_tfidf_tuning_rows)
best_weighted_tfidf_row = weighted_tfidf_tuning_df.loc[
    weighted_tfidf_tuning_df[CONFIG["primary_metric"]].idxmax()
]
BEST_WEIGHT_LABEL = str(best_weighted_tfidf_row["weight_label"])
BEST_WEIGHTED_FIELD_WEIGHTS = dict(
    CONFIG["weighted_field_weight_candidates"][BEST_WEIGHT_LABEL]
)

print(
    f"Selected Weighted-TFIDF configuration={BEST_WEIGHT_LABEL} using validation "
    f"{CONFIG['primary_metric']}={best_weighted_tfidf_row[CONFIG['primary_metric']]:.6f}"
)
display(
    weighted_tfidf_tuning_df[
        ["weight_label", "Hit@10", "NDCG@10", "MRR", "field_weights"]
    ].sort_values(CONFIG["primary_metric"], ascending=False)
)


[Weighted-TFIDF-title2_categories2] item_matrix shape=(81322, 32768)                                
[Weighted-TFIDF-title2_categories2] validation: Hit@10=0.5845 NDCG@10=0.4293 MRR=0.3972             
[Weighted-TFIDF-title3_categories2] item_matrix shape=(81322, 32768)                                
[Weighted-TFIDF-title3_categories2] validation: Hit@10=0.5881 NDCG@10=0.4326 MRR=0.4001             
[Weighted-TFIDF-title2_categories3] item_matrix shape=(81322, 32768)                                
[Weighted-TFIDF-title2_categories3] validation: Hit@10=0.5904 NDCG@10=0.4312 MRR=0.3978             
[Weighted-TFIDF-title2_categories2_features1_5] item_matrix shape=(81322, 32768)                    
[Weighted-TFIDF-title2_categories2_features1_5] validation: Hit@10=0.5756 NDCG@10=0.4217 MRR=0.3900 
[Weighted-TFIDF-title2_categories2_5] item_matrix shape=(81322, 32768)                              
[Weighted-TFIDF-title2_categories2_5] validation: Hit@10=0.5885 NDCG@10=0.4308 MRR=0.3978  

,weight_label,Hit@10,NDCG@10,MRR,field_weights
13,title2_75_categories3,0.593152,0.434093,0.400600,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
12,title2_5_categories3_25,0.593507,0.433432,0.399660,"{""author"": 0.0, ""categories"": 3.25, ""descripti..."
8,title2_5_categories3,0.592628,0.433400,0.399878,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
11,title2_5_categories2_75,0.591965,0.433367,0.400014,"{""author"": 0.0, ""categories"": 2.75, ""descripti..."
1,title3_categories2,0.588111,0.432551,0.400107,"{""author"": 0.0, ""categories"": 2.0, ""descriptio..."
10,title2_25_categories3,0.591503,0.432377,0.398953,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
9,title2_25_categories2_75,0.590855,0.432288,0.399001,"{""author"": 0.0, ""categories"": 2.75, ""descripti..."
2,title2_categories3,0.590439,0.431244,0.397839,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
5,title2_categories3_5,0.591718,0.431111,0.397294,"{""author"": 0.0, ""categories"": 3.5, ""descriptio..."
4,title2_categories2_5,0.588450,0.430806,0.397847,"{""author"": 0.0, ""categories"": 2.5, ""descriptio..."


### 5.26 SVD Dimension Tuning for Field-Weighted TF-IDF

Compare different SVD dimensions on the validation set while using the previously selected field weights and keeping other model Settings unchanged. The dimension with the highest score of the main validation metric will be saved as `BEST WEIGHTED SVD DIM`.

After that, the best-performing field-weighted SVD model was compared with the fine-tuned field-weighted TF-IDF model without dimensionality reduction. To determine whether SVD could further improve performance.

In [97]:
weighted_svd_tuning_rows = []
for n_components in CONFIG["weighted_svd_dim_candidates"]:
    _, result = run_variant(
        f"Weighted-SVD-dim{n_components}",
        FieldWeightedSVDContentVectorizer(
            CONFIG["basic_n_features"],
            BEST_WEIGHTED_FIELD_WEIGHTS,
            CONFIG["description_truncate_chars"],
            n_components,
            ngram_range=CONFIG["weighted_tfidf_ngram_range"],
            sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
        ),
        items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
    )
    row = result[0]
    row["n_components"] = n_components
    weighted_svd_tuning_rows.append(row)

weighted_svd_tuning_df = pd.DataFrame(weighted_svd_tuning_rows)
best_weighted_svd_row = weighted_svd_tuning_df.loc[
    weighted_svd_tuning_df[CONFIG["primary_metric"]].idxmax()
]
BEST_WEIGHTED_SVD_DIM = int(best_weighted_svd_row["n_components"])

no_reduction_score = float(best_weighted_tfidf_row[CONFIG["primary_metric"]])
best_reduced_score = float(best_weighted_svd_row[CONFIG["primary_metric"]])
verdict = (
    "SVD HELPS on top of field-weighting"
    if best_reduced_score > no_reduction_score
    else "SVD still hurts, even on the field-weighted representation"
)

print(
    f"Best SVD dim on field-weighted counts={BEST_WEIGHTED_SVD_DIM} using validation "
    f"{CONFIG['primary_metric']}={best_reduced_score:.6f} "
    f"(no-reduction Weighted-TFIDF-TUNED={no_reduction_score:.6f}) -> {verdict}"
)
display(
    weighted_svd_tuning_df[["n_components", "Hit@10", "NDCG@10", "MRR"]].sort_values(
        CONFIG["primary_metric"], ascending=False
    )
)


[Weighted-SVD-dim128] item_matrix shape=(81322, 128)                                                
[Weighted-SVD-dim128] validation: Hit@10=0.5919 NDCG@10=0.3863 MRR=0.3392                           
[Weighted-SVD-dim256] item_matrix shape=(81322, 256)                                                
[Weighted-SVD-dim256] validation: Hit@10=0.6005 NDCG@10=0.4008 MRR=0.3550                           
[Weighted-SVD-dim512] item_matrix shape=(81322, 512)                                                
[Weighted-SVD-dim512] validation: Hit@10=0.6065 NDCG@10=0.4141 MRR=0.3702                           
Best SVD dim on field-weighted counts=512 using validation NDCG@10=0.414140 (no-reduction Weighted-TFIDF-TUNED=0.434093) -> SVD still hurts, even on the field-weighted representation


,n_components,Hit@10,NDCG@10,MRR
2,512,0.606503,0.414140,0.370217
1,256,0.600490,0.400802,0.355050
0,128,0.591934,0.386322,0.339180


## 5.3 Final Model Selection and Test Evaluation

In [98]:
CONFIG["svd_n_components"] = BEST_SVD_DIM
CONFIG["verified_purchase_multiplier"] = BEST_VERIFIED_MULTIPLIER

tuned_svd_items_df = items_df.copy()
tuned_svd_items_df["content_text"] = build_content_text(
    tuned_svd_items_df,
    CONFIG["description_truncate_chars"],
    include_author=BEST_INCLUDE_AUTHOR,
)

tuned_svd_val_recommender, tuned_svd_val_result = run_variant(
    "TFIDF-SVD-TUNED",
    SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    ),
    tuned_svd_items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=False,
    eval_splits=("validation",),
    verified_purchase_multiplier=BEST_VERIFIED_MULTIPLIER,
)

weighted_tfidf_val_recommender, weighted_tfidf_val_result = run_variant(
    "Weighted-TFIDF-TUNED",
    FieldWeightedTFIDFVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    ),
    items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=True,
    eval_splits=("validation",),
)

weighted_svd_val_recommender, weighted_svd_val_result = run_variant(
    "Weighted-SVD-TUNED",
    FieldWeightedSVDContentVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        BEST_WEIGHTED_SVD_DIM,
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    ),
    items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=False,
    eval_splits=("validation",),
)

family_validation_rows = results_df[
    (results_df["split"] == "validation")
    & (results_df["model"].isin([
        "Basic-TFIDF",
    ]))
].copy()
family_validation_rows = pd.concat(
    [
        family_validation_rows,
        pd.DataFrame(tuned_svd_val_result),
        pd.DataFrame(weighted_tfidf_val_result),
        pd.DataFrame(weighted_svd_val_result),
    ],
    ignore_index=True,
)
best_family_row = family_validation_rows.loc[
    family_validation_rows[CONFIG["primary_metric"]].idxmax()
]
SELECTED_CONTENT_MODEL = str(best_family_row["model"])

print(
    f"Locked final content model={SELECTED_CONTENT_MODEL} using validation "
    f"{CONFIG['primary_metric']}={best_family_row[CONFIG['primary_metric']]:.6f}"
)
display(
    family_validation_rows.sort_values(
        CONFIG["primary_metric"],
        ascending=False,
    )
)

if SELECTED_CONTENT_MODEL == "Basic-TFIDF":
    final_vectorizer = TFIDFHashingVectorizer(CONFIG["basic_n_features"])
    final_items_df = items_df
    final_use_lazy_profiles = True
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "Weighted-TFIDF-TUNED":
    final_vectorizer = FieldWeightedTFIDFVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    )
    final_items_df = items_df
    final_use_lazy_profiles = True
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "Weighted-SVD-TUNED":
    final_vectorizer = FieldWeightedSVDContentVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        BEST_WEIGHTED_SVD_DIM,
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    )
    final_items_df = items_df
    final_use_lazy_profiles = False
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "TFIDF-SVD-TUNED":
    final_vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    )
    final_items_df = tuned_svd_items_df
    final_use_lazy_profiles = False
    final_verified_multiplier = BEST_VERIFIED_MULTIPLIER
else:
    raise ValueError(f"Unsupported selected model: {SELECTED_CONTENT_MODEL}")

final_recommender, final_test_results = run_variant(
    f"{SELECTED_CONTENT_MODEL}-FINAL",
    final_vectorizer,
    final_items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=final_use_lazy_profiles,
    eval_splits=("test",),
    verified_purchase_multiplier=final_verified_multiplier,
)

diagnostic_rng = np.random.default_rng(CONFIG["evaluation_seed"])
diagnostic_users = diagnostic_rng.choice(
    np.arange(n_users),
    size=min(CONFIG["diagnostic_n_users"], n_users),
    replace=False,
)

final_catalogue_diagnostics_df, final_catalogue_metrics = evaluate_catalogue_diagnostics(
    final_recommender,
    diagnostic_users,
    n_items,
    CONFIG["top_k"],
    train_item_counts,
)
final_catalogue_metrics["model"] = f"{SELECTED_CONTENT_MODEL}-FINAL"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_candidate_set_results(to_report_names(pd.DataFrame(final_test_results)))

final_catalogue_diagnostics_df.to_csv(
    OUTPUT_DIR / "content_based_catalogue_diagnostics.csv", index=False
)
with open(OUTPUT_DIR / "content_based_catalogue_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(final_catalogue_metrics, handle, indent=2)

with open(OUTPUT_DIR / "final_locked_test_results.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "selected_by": CONFIG["primary_metric"],
            "selected_model": SELECTED_CONTENT_MODEL,
            "validation_candidates": family_validation_rows.to_dict(orient="records"),
            "locked_configuration": {
                "svd_dim": BEST_SVD_DIM,
                "weighted_svd_dim": BEST_WEIGHTED_SVD_DIM,
                "include_author": BEST_INCLUDE_AUTHOR,
                "verified_purchase_multiplier": BEST_VERIFIED_MULTIPLIER,
                "model_seed": CONFIG["model_seed"],
                "evaluation_seed": CONFIG["evaluation_seed"],
                "validation_context": "train",
                "test_context": "train+validation",
            },
            "test_results": final_test_results,
            "catalogue_diagnostics": final_catalogue_metrics,
        },
        handle,
        indent=2,
    )

print("Final test result (1 positive + 100 fixed negatives):")
display(pd.DataFrame(final_test_results))
print("Full-catalogue diagnostics on the shared deterministic user sample:")
display(pd.DataFrame([final_catalogue_metrics]))
print("Per-user catalogue diagnostics (first 5 rows):")
display(final_catalogue_diagnostics_df.head())


[TFIDF-SVD-TUNED] item_matrix shape=(81322, 256)                                                    
[TFIDF-SVD-TUNED] validation: Hit@10=0.5895 NDCG@10=0.3974 MRR=0.3541                               
[Weighted-TFIDF-TUNED] item_matrix shape=(81322, 32768)                                             
[Weighted-TFIDF-TUNED] validation: Hit@10=0.5932 NDCG@10=0.4341 MRR=0.4006                          
[Weighted-SVD-TUNED] item_matrix shape=(81322, 512)                                                 
[Weighted-SVD-TUNED] validation: Hit@10=0.6065 NDCG@10=0.4141 MRR=0.3702                            
Locked final content model=Weighted-TFIDF-TUNED using validation NDCG@10=0.434093


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
2,Weighted-TFIDF-TUNED,validation,0.593152,0.593152,0.059315,0.906863,0.434093,0.400600,64867
3,Weighted-SVD-TUNED,validation,0.606503,0.606503,0.060650,0.906995,0.414140,0.370217,64867
1,TFIDF-SVD-TUNED,validation,0.589499,0.589499,0.058950,0.906827,0.397368,0.354133,64867
0,Basic-TFIDF,validation,0.537484,0.537484,0.053748,0.906312,0.389874,0.361362,64867


[Weighted-TFIDF-TUNED-FINAL] item_matrix shape=(81322, 32768)                                       
[Weighted-TFIDF-TUNED-FINAL] test: Hit@10=0.5914 NDCG@10=0.4283 MRR=0.3935                          
Final test result (1 positive + 100 fixed negatives):


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
0,Weighted-TFIDF-TUNED-FINAL,test,0.591441,0.591441,0.059144,0.906846,0.428251,0.393531,64867


Full-catalogue diagnostics on the shared deterministic user sample:


,users,catalogue_coverage,unique_recommended_items,model
0,5000,0.248137,20179,Weighted-TFIDF-TUNED-FINAL


Per-user catalogue diagnostics (first 5 rows):


,user_idx,novelty,intra_list_diversity,avg_train_popularity
0,39754,15.563100,0.755556,85.0
1,15519,15.675596,0.688889,26.4
2,26537,16.808202,0.044444,8.1
3,17169,14.543150,0.000000,111.1
4,60483,15.695243,0.155556,20.2


Save the finally selected Content-Based model as a file

In [99]:
import joblib

MODEL_HANDOFF_DIR = OUTPUT_DIR


def save_content_based_handoff(vectorizer, model_name: str, locked_configuration: dict) -> dict:
    vectorizer_path = MODEL_HANDOFF_DIR / "content_based_final_vectorizer.joblib"

    detached_nn_index, vectorizer.nn_index = vectorizer.nn_index, None
    try:
        joblib.dump(vectorizer, vectorizer_path)
    finally:
        vectorizer.nn_index = detached_nn_index

    item_matrix = vectorizer.item_matrix
    is_sparse = sp.issparse(item_matrix)
    if is_sparse:
        embeddings_path = MODEL_HANDOFF_DIR / "content_based_item_embeddings.npz"
        sp.save_npz(embeddings_path, item_matrix.tocsr())
    else:
        embeddings_path = MODEL_HANDOFF_DIR / "content_based_item_embeddings.npy"
        np.save(embeddings_path, np.asarray(item_matrix))

    item_idx_path = MODEL_HANDOFF_DIR / "content_based_item_idx_order.npy"
    np.save(item_idx_path, np.asarray(vectorizer.item_idx_order))

    model_card = {
        "selected_model": model_name,
        "embedding_dim": int(item_matrix.shape[1]),
        "n_items": int(item_matrix.shape[0]),
        "is_sparse": bool(is_sparse),
        "similarity_metric": "cosine (vectors are L2-normalised, so dot product == cosine similarity)",
        "vectorizer_file": vectorizer_path.name,
        "embeddings_file": embeddings_path.name,
        "item_idx_file": item_idx_path.name,
        "locked_configuration": locked_configuration,
        "how_to_use": {
            "score_new_item_text": (
                "vectorizer = joblib.load('content_based_final_vectorizer.joblib'); "
                "requires this notebook's vectorizer classes to already be defined; call "
                "vectorizer.build_index(n_neighbors=...) once before query_topk() (the "
                "fitted index was deliberately not saved -- see markdown note); then "
                "vectorizer.transform_new_items(...) to embed a brand-new item's text."
            ),
            "reuse_embeddings_only": (
                "item_matrix = scipy.sparse.load_npz(...) if is_sparse else np.load(...); "
                "item_idx_order = np.load('content_based_item_idx_order.npy'); "
                "row = {idx: i for i, idx in enumerate(item_idx_order)}[item_idx]; "
                "score = item_matrix[row] @ user_profile_vector -- no custom classes needed."
            ),
        },
    }
    model_card_path = MODEL_HANDOFF_DIR / "content_based_model_card.json"
    with open(model_card_path, "w", encoding="utf-8") as handle:
        json.dump(model_card, handle, indent=2)

    return model_card


final_model_card = save_content_based_handoff(
    final_vectorizer,
    SELECTED_CONTENT_MODEL,
    {
        "svd_dim": BEST_SVD_DIM,
        "weighted_svd_dim": BEST_WEIGHTED_SVD_DIM,
        "include_author": BEST_INCLUDE_AUTHOR,
        "verified_purchase_multiplier": BEST_VERIFIED_MULTIPLIER,
        "field_weights": (
            BEST_WEIGHTED_FIELD_WEIGHTS
            if SELECTED_CONTENT_MODEL in {"Weighted-TFIDF-TUNED", "Weighted-SVD-TUNED"}
            else None
        ),
    },
)

print("Saved content-based handoff artifacts to", OUTPUT_DIR.resolve())
for key in ("vectorizer_file", "embeddings_file", "item_idx_file"):
    print(f"  outputs/{final_model_card[key]}")
print("  outputs/content_based_model_card.json")


Saved content-based handoff artifacts to /Users/lumi/PycharmProjects/PythonProject/comp9727/teamProject/outputs/content_based
  outputs/content_based_final_vectorizer.joblib
  outputs/content_based_item_embeddings.npz
  outputs/content_based_item_idx_order.npy
  outputs/content_based_model_card.json


Export all user–candidate item scores produced by the selected Content-Based model on the validation and test sets, and save them as a compressed CSV file for subsequent score-level fusion in the Hybrid recommender.

In [100]:
def export_candidate_scores(
    recommender,
    positives_df: pd.DataFrame,
    negatives_dict: dict[int, list[int]],
    split: str,
) -> pd.DataFrame:
    split = normalise_split_name(split)
    n_users = len(positives_df)
    width = 101
    user_col = np.empty(n_users * width, dtype=np.int32)
    item_col = np.empty(n_users * width, dtype=np.int32)
    score_col = np.empty(n_users * width, dtype=np.float32)
    positive_col = np.zeros(n_users * width, dtype=bool)
    for position, row in enumerate(positives_df.itertuples()):
        user_idx = int(row.user_idx)
        positive = int(row.pos_idx)
        candidates = np.asarray([positive, *negatives_dict[user_idx]], dtype=np.int32)
        start, end = position * width, (position + 1) * width
        user_col[start:end] = user_idx
        item_col[start:end] = candidates
        score_col[start:end] = recommender.score_candidates(user_idx, candidates)
        positive_col[start] = True
    return pd.DataFrame({'user_idx': user_col, 'item_idx': item_col, 'split': split,
                         'score': score_col, 'is_positive': positive_col})

_validation_recommender_by_model = {
    "Basic-TFIDF": recommenders.get("Basic-TFIDF"),
    "TFIDF-SVD-TUNED": globals().get("tuned_svd_val_recommender"),
    "Weighted-TFIDF-TUNED": globals().get("weighted_tfidf_val_recommender"),
    "Weighted-SVD-TUNED": globals().get("weighted_svd_val_recommender"),
}
selected_validation_recommender = _validation_recommender_by_model[SELECTED_CONTENT_MODEL]
if selected_validation_recommender is None:
    raise RuntimeError(
        f"No validation-context recommender found for {SELECTED_CONTENT_MODEL!r} -- "
        "make sure the cells above (run_all_variants() and/or the FINAL LOCK-AND-TEST "
        "CELL) have already run in this session."
    )

validation_scores_df = export_candidate_scores(
    selected_validation_recommender,
    load_eval_positives(DATA_DIR, "validation"),
    load_eval_negatives(DATA_DIR, "validation"),
    "validation",
)
test_scores_df = export_candidate_scores(
    final_recommender,
    load_eval_positives(DATA_DIR, "test"),
    load_eval_negatives(DATA_DIR, "test"),
    "test",
)

candidate_scores_df = pd.concat([validation_scores_df, test_scores_df], ignore_index=True)
candidate_scores_path = OUTPUT_DIR / "content_based_candidate_scores.csv.gz"
candidate_scores_df.to_csv(candidate_scores_path, index=False, compression="gzip")

print(f"Exported {len(candidate_scores_df):,} (user, candidate) score rows for model={SELECTED_CONTENT_MODEL}")
print(f"  -> {candidate_scores_path}")
display(candidate_scores_df.head())


Exported 13,103,134 (user, candidate) score rows for model=Weighted-TFIDF-TUNED
  -> /Users/lumi/PycharmProjects/PythonProject/comp9727/teamProject/outputs/content_based/content_based_candidate_scores.csv.gz


,user_idx,item_idx,split,score,is_positive
0,0,2617,validation,0.071234,True
1,0,5053,validation,0.036460,False
2,0,37497,validation,0.033347,False
3,0,18865,validation,0.026172,False
4,0,60625,validation,0.020687,False


### 5.4 Content-Based Model Summary
Among the four content-based recommendation models, Weighted-TFIDF-TUNED achieved the highest validation NDCG@10 of 0.4341 and the highest MRR of 0.4006. Although Weighted-SVD-TUNED achieved a slightly higher Hit@10, its lower NDCG@10. That suggests the relevant books were generally ranked lower within the Top-10 recommendation list. Therefore, based on the predefined primary evaluation metric, NDCG@10, Weighted-TFIDF-TUNED was selected as the final content-based model.